# Data preparation of generation data from ENTSO-E transparency

source: https://transparency.entsoe.eu/

Creates the following parsed datasets

- Hourly generation per technology and country for selected year (generation_"+year+"_hourly_entsoe.csv) 
- Weekly generation per technology and country for selected year (generation_"+year+"_weekly_entsoe.csv) 
- Monthly generation per technology and country for selected year (generation_"+year+"_monthly_entsoe.csv) 
- Yearly generation per technology and country for selected year (generation_"+year+"_annual_entsoe.csv) 

Settings in next window

In [1]:
#download files again (yes/no)?
download = "no"

#set year for data creation
year = '2024'

In [2]:
import pysftp
import sys
import os
import pandas as pd
import datetime as dt
import plotly.express as px

c:\Users\jonas\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,
c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
#ENTSO-E sftp settings
host = "sftp-transparency.entsoe.eu"
#enter your password here
password = "3?mJ}V4us?!L}5E"                
#enter email address here
username = "jsavelsberg@ethz.ch"                
port = '22'

In [4]:
cnopts = pysftp.CnOpts()
cnopts.hostkeys = None

c:\Users\jonas\anaconda3\Lib\site-packages\pysftp\__init__.py:61: UserWarning: Failed to load HostKeys from C:\Users\jonas\.ssh\known_hosts.  You will need to explicitly load HostKeys (cnopts.hostkeys.load(filename)) or disableHostKey checking (cnopts.hostkeys = None).
  warnings.warn(wmsg, UserWarning)


In [5]:
dir_out = "../parsed_data/"

In [6]:
#technology definition
dict_agg_tech = {"Other": "Other",
                "Wind Offshore": "WindOffshore",
                "Fossil Brown coal/Lignite": "Lignite",
                "Nuclear": "Nuclear",
                "Fossil Hard coal": "HardCoal",
                "Geothermal": "Other",
                "Fossil Coal-derived gas": "Other",
                "Hydro Pumped Storage": "Pump",
                "Hydro Run-of-river and poundage": "RunOfRiver",
                "Biomass": "Biomass",
                "Fossil Peat": "Other",
                "Fossil Oil shale": "Oil",
                "Fossil Oil": "Oil",
                "Hydro Water Reservoir": "Reservoir",
                "Marine": "Other",
                "Wind Onshore": "WindOnshore",
                "Other renewable": "Other",
                "Solar": "Solar",
                "Waste": "Other",
                "Fossil Gas": "Gas",
                "rooftop_pv" : "Solar",
                "onshore_wind" : "WindOnshore",
                "offshore_wind" : "WindOffshore"}

In [7]:
#Connect to ENTSO-E Transparency FTP
#for this to work, pysftp 0.2.8 is needed!
path = '/TP_export/'
path_local = os.path.join(os.pardir,'source_data/ENTSOE/') 

In [10]:
# show list of all available folders (uncomment last line if needed)
# relevant folder was renamed to AggregatedGenerationPerType_16.1.B_C
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        files = sftp.listdir('/TP_export/')   
        print(files)

In [17]:
#load file names from server
path_gen = path+'AggregatedGenerationPerType_16.1.B_C/'
path_gen_local = path_local+'generation/'
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        print("Connection succesfully established.")
        # show list of files
        files = sftp.listdir(path_gen)
        
if download =="no": 
    files = os.listdir(path_gen_local)
if year != "":
            files = [i for i in files if year in i]
print(files)

['2024_01_AggregatedGenerationPerType_16.1.B_C.csv', '2024_02_AggregatedGenerationPerType_16.1.B_C.csv', '2024_03_AggregatedGenerationPerType_16.1.B_C.csv', '2024_04_AggregatedGenerationPerType_16.1.B_C.csv', '2024_05_AggregatedGenerationPerType_16.1.B_C.csv', '2024_06_AggregatedGenerationPerType_16.1.B_C.csv', '2024_07_AggregatedGenerationPerType_16.1.B_C.csv', '2024_08_AggregatedGenerationPerType_16.1.B_C.csv', '2024_09_AggregatedGenerationPerType_16.1.B_C.csv', '2024_10_AggregatedGenerationPerType_16.1.B_C.csv', '2024_11_AggregatedGenerationPerType_16.1.B_C.csv', '2024_12_AggregatedGenerationPerType_16.1.B_C.csv']


In [18]:
#download aggregated generation data (AggregatedGenerationPerType)
if download == "yes":
    with pysftp.Connection(host=host, username=username, password=password, cnopts=cnopts) as sftp:
        for file in files:
            sftp.get(path_gen+file,path_gen_local+file)
            print('Successfully downloaded file '+file)

In [19]:
#combine files to one data frame
df_gen_in = pd.DataFrame()
for file in files:
    df_temp = pd.read_csv(path_gen_local+file,
                          decimal=".",sep="\t",
                          parse_dates=True, index_col="DateTime")
    df_gen_in = pd.concat([df_gen_in,df_temp])
#country values for HR are missing so we change this to area type code
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'CTY')),'AreaTypeCode'] = 'not_it'
df_gen_in.loc[((df_gen_in.MapCode == 'HR') & (df_gen_in.AreaTypeCode == 'BZN')),'AreaTypeCode'] = 'CTY'
df_gen_in = df_gen_in[df_gen_in.AreaTypeCode == "CTY"].reset_index()
df_gen_in = df_gen_in.sort_values(by=['DateTime'])
df_gen_in["technology"] = df_gen_in.ProductionType.map(dict_agg_tech)
df_gen_in = df_gen_in.drop(["AreaCode","AreaTypeCode","ResolutionCode","AreaName","ProductionType","UpdateTime"], axis=1)
df_gen_in.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6399949 entries, 0 to 6327524
Data columns (total 5 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   DateTime                datetime64[ns]
 1   MapCode                 object        
 2   ActualGenerationOutput  float64       
 3   ActualConsumption       float64       
 4   technology              object        
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 293.0+ MB


In [20]:
#Aggregate technologies:
df_gen = df_gen_in.groupby(["DateTime", "MapCode", "technology"], as_index=False).sum()
df_gen.columns = ["date", "country", "tech", "output", "demand"]
df_gen["net_generation"] = df_gen.output - df_gen.demand
df_gen.info

<bound method DataFrame.info of                        date country         tech   output  demand  \
0       2024-01-01 00:00:00      AT      Biomass   144.00     0.0   
1       2024-01-01 00:00:00      AT          Gas    32.80     0.0   
2       2024-01-01 00:00:00      AT     HardCoal     0.00     0.0   
3       2024-01-01 00:00:00      AT          Oil     0.00     0.0   
4       2024-01-01 00:00:00      AT        Other   122.07     0.0   
...                     ...     ...          ...      ...     ...   
5362581 2024-12-31 23:45:00      RO      Nuclear  1272.00     0.0   
5362582 2024-12-31 23:45:00      RO    Reservoir    93.00     0.0   
5362583 2024-12-31 23:45:00      RO   RunOfRiver   775.00     0.0   
5362584 2024-12-31 23:45:00      RO        Solar     0.00     0.0   
5362585 2024-12-31 23:45:00      RO  WindOnshore   709.00     0.0   

         net_generation  
0                144.00  
1                 32.80  
2                  0.00  
3                  0.00  
4        

In [21]:
df_gen = df_gen.set_index(['date','country','tech'])
df_gen.head()

output  demand  net_generation
date       country tech                                    
2024-01-01 AT      Biomass   144.00     0.0          144.00
                   Gas        32.80     0.0           32.80
                   HardCoal    0.00     0.0            0.00
                   Oil         0.00     0.0            0.00
                   Other     122.07     0.0          122.07

In [22]:
#some values are reported quarter hourly so we have to resample to hourly values
df_gen_hourly = df_gen.groupby([pd.Grouper(level='country'),
                                pd.Grouper(level='tech'), 
                                pd.Grouper(level='date', freq='1h')]
                               ).mean()
df_gen_hourly.head()

output  demand  net_generation
country tech    date                                               
AT      Biomass 2024-01-01 00:00:00   143.0     0.0           143.0
                2024-01-01 01:00:00   140.0     0.0           140.0
                2024-01-01 02:00:00   140.0     0.0           140.0
                2024-01-01 03:00:00   140.0     0.0           140.0
                2024-01-01 04:00:00   140.0     0.0           140.0

In [23]:
#We are primarily interested in renewable data. Let's see how complete they are
df_ = df_gen_hourly.groupby(["country", "tech"]).net_generation.count()
df_ = df_.reset_index().pivot_table(index=["country"], columns="tech", values="net_generation")
df_.loc[(slice(None)), ["Solar", "WindOnshore", "WindOffshore", "RunOfRiver","Biomass"]]

tech,Solar,WindOnshore,WindOffshore,RunOfRiver,Biomass
country,,,,,
AT,8784.0,8784.0,NaN,8784.0,8784.0
BA,7849.0,7849.0,NaN,7849.0,NaN
BE,8784.0,8784.0,8784.0,8784.0,8784.0
BG,8784.0,8784.0,NaN,8784.0,8784.0
CH,8784.0,8784.0,NaN,8784.0,NaN
CZ,8783.0,8784.0,NaN,8784.0,8784.0
DE,8784.0,8784.0,8784.0,8784.0,8784.0
DK,8783.0,8784.0,8783.0,NaN,8783.0
EE,8784.0,8783.0,NaN,8783.0,8783.0


In [24]:
#export annual and monthly data for checking of the data quality
df_gen_a = df_gen_hourly.groupby(["country", "tech"]).sum()/1000000
df_gen_a.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 293 entries, ('AT', 'Biomass') to ('XK', 'Lignite')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          293 non-null    float64
 1   demand          293 non-null    float64
 2   net_generation  293 non-null    float64
dtypes: float64(3)
memory usage: 7.9+ KB


In [25]:
df_gen_m = df_gen_hourly.groupby([pd.Grouper(freq='ME', level='date'), "country", "tech"]).sum()
df_gen_m.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3484 entries, (Timestamp('2024-01-31 00:00:00'), 'AT', 'Biomass') to (Timestamp('2024-12-31 00:00:00'), 'XK', 'Lignite')
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   output          3484 non-null   float64
 1   demand          3484 non-null   float64
 2   net_generation  3484 non-null   float64
dtypes: float64(3)
memory usage: 92.5+ KB


In [26]:
#we also export weekly values
df_gen_w = df_gen_hourly.groupby([pd.Grouper(freq='W', level='date'), "country", "tech"]).sum()
df_gen_w.head()

output  demand  net_generation
date       country tech                                       
2024-01-07 AT      Biomass    24453.00     0.0        24453.00
                   Gas       123265.00     0.0       123265.00
                   HardCoal       0.00     0.0            0.00
                   Oil            0.00     0.0            0.00
                   Other      20507.76     0.0        20507.76

In [27]:
df_gen_hourly.to_csv(dir_out + "generation_"+year+"_hourly_entsoe.csv", index=True)
df_gen_w.to_csv(dir_out + "generation_"+year+"_weekly_entsoe.csv", index=True)
df_gen_a.to_csv(dir_out + "generation_"+year+"_annual_entsoe.csv", index=True)
df_gen_m.to_csv(dir_out + "generation_"+year+"_monthly_entsoe.csv", index=True)

In [28]:
df_gen.reset_index().country.unique()

array(['AT', 'BA', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI',
       'FR', 'GB', 'GE', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV',
       'MD', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI',
       'SK', 'XK'], dtype=object)